In [46]:
import os, rasterio, sys, rioxarray, pyflwdir, shutil, cdsapi, dotenv, calendar
sys.path.append('backend/app/')
from rasterio.features import rasterize
from dateutil.relativedelta import relativedelta
from rasterio.mask import mask
from datetime import datetime, timedelta
from netCDF4 import Dataset, date2num
import geopandas as gpd, pandas as pd
import numpy as np, xarray as xr
from backend.app.services import flow_functions
from shapely.geometry import Polygon, MultiPolygon
from shapely import force_2d
from pyflwdir import dem
from scipy.ndimage import sobel
from hydromt_wflow import WflowSbmModel
from tqdm import tqdm
from pathlib import Path
np.random.seed(42)

In [ ]:
# CDS_URL=https://cds.climate.copernicus.eu/api
# CDS_API_KEY=4ee95413-088c-43a2-a572-95a90f6b2e40

In [5]:
def keep_polygon(geom):
    if geom.geom_type == 'GeometryCollection':
        polys = [g for g in geom.geoms if isinstance(g, (Polygon, MultiPolygon))]
        if len(polys) == 0: return None
        return polys[0]
    return geom

def fix_invalid_polygon(gdf, cols):
    gdf_new, name = gdf.copy(), cols[0]
    gdf_valid, gdf_nan = gdf_new[gdf_new[name] != ''], gdf_new[gdf_new[name] == '']
    if gdf_nan.shape[0] > 0:
        gdf_valid['geometry'] = gdf_valid['geometry'].apply(keep_polygon)
        gdf_nan['geometry'] = gdf_nan['geometry'].apply(keep_polygon)
        # Spatial join nearest
        gdf_filled = gpd.sjoin_nearest(
            gdf_nan, gdf_valid[['geometry', name]], how='left', distance_col='dist'
        )
        gdf_filled = gdf_filled.drop_duplicates(subset='_id')
        gdf_new.loc[gdf_filled.index, cols] = gdf_valid.loc[gdf_filled['index_right'], cols].values
    gdf_new['geometry'] = gdf_new['geometry'].apply(keep_polygon)
    return gdf_new

def clip_catchment(catchment, terrain):
    clipped = terrain.rio.clip(catchment.geometry, catchment.crs, drop=False)
    clipped = clipped.fillna(-9999)
    clipped.rio.write_nodata(-9999, inplace=True)
    return clipped

def write_tif(path, terrain, geo, col=''):
    transform = terrain.rio.transform()
    if col == '': shapes = ((geom, 1) for geom in geo.geometry)
    else: shapes = ((geom, value) for geom, value in zip(geo.geometry, geo[col]))
    raster = rasterize(
        shapes=shapes, out_shape=(terrain.rio.height, terrain.rio.width),
        transform=transform, fill=-9999, dtype="float32", all_touched=True
    )
    meta = {
        "driver": "GTiff", "height": terrain.rio.height,
        "width": terrain.rio.width, "count": 1, "dtype": "float32", 
        "crs": terrain.rio.crs, "transform": transform, "nodata": -9999
    }
    with rasterio.open(path, "w", **meta) as dst:
        dst.write(raster, 1)

def write_geotiff(data, profile, output_path):
    profile.update(dtype=data.dtype, count=1, compress='lzw')
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(data, 1)

In [6]:
sample_folder, test_folder = 'inputs', 'test'
catchment_path = os.path.join(sample_folder, 'catchment.geojson')
terrain_path = os.path.join(sample_folder, 'dtm10.tif')
soil_path = os.path.join(sample_folder, 'soil.geojson')
land_path = os.path.join(sample_folder, 'land.geojson')
river_path = os.path.join(sample_folder, 'river.geojson')
terrain = rioxarray.open_rasterio(terrain_path).squeeze()
catchment = gpd.read_file(catchment_path)
initial_param = [0.25,0.3,50,0,0,500,0,0,100]
soil = gpd.read_file(soil_path)
land = gpd.read_file(land_path)
river = gpd.read_file(river_path)

## Create merit hydro data

In [ ]:
with rasterio.open(terrain_path) as src:
    dem_array = src.read(1).astype(np.float32)
    profile = src.profile
    transform = src.transform
    crs = src.crs
NODATA_DEM, NODATA_INT = -9999.0, 0
profile.update(dtype=np.float32, nodata=NODATA_DEM)
# Fill depressions
elevtn_array, flwdir_array = dem.fill_depressions(
    elevtn=dem_array, nodata=NODATA_DEM, max_depth=-1
)
elevtn_array = np.where(np.isfinite(elevtn_array), elevtn_array, NODATA_DEM)
# Create flow direction
flw = pyflwdir.from_array(
    data=flwdir_array, ftype='d8', transform=transform, 
    latlon=crs.is_geographic
)
# Create slope
dx, dy = transform.a, abs(transform.e)
# Gradient elevation
dzdx = sobel(elevtn_array, axis=1, mode='nearest') / (8 * dx)
dzdy = sobel(elevtn_array, axis=0, mode='nearest') / (8 * dy)
slope_array = np.sqrt(dzdx**2 + dzdy**2)
slope_array = np.where(elevtn_array == NODATA_DEM, NODATA_DEM, slope_array).astype(np.float32)
# Create basins
basins_array = flw.basins()
# Create stream order
uparea_array = flw.upstream_area(unit='km2')
# Create stream mask and stream order
stream_mask = uparea_array > 30
strord_array = flw.stream_order(type='strahler', mask=stream_mask)
merit_dir = os.path.normpath(f'{test_folder}/data/merit_hydro')
if not os.path.exists(merit_dir): os.makedirs(merit_dir)
write_geotiff(elevtn_array, profile, os.path.join(merit_dir, 'elevtn.tif'))
profile_flwdir = {**profile, 'dtype': np.uint8, 'nodata': NODATA_INT}
write_geotiff(flwdir_array, profile_flwdir, os.path.join(merit_dir, 'flwdir.tif'))
profile_slope = {**profile, 'dtype': np.float32, 'nodata': NODATA_DEM}
write_geotiff(slope_array, profile_slope, os.path.join(merit_dir, 'lndslp.tif'))
profile_basins = {**profile, 'dtype': np.int32, 'nodata': NODATA_INT}
write_geotiff(basins_array, profile_basins, os.path.join(merit_dir, 'basins.tif'))
profile_uparea = {**profile, 'dtype': np.float32, 'nodata': NODATA_DEM}
write_geotiff(uparea_array, profile_uparea, os.path.join(merit_dir, 'uparea.tif'))
profile_strord = {**profile, 'dtype': np.int16, 'nodata': NODATA_INT}
write_geotiff(strord_array, profile_strord, os.path.join(merit_dir, 'strord.tif'))

## Prepare forcing data from the global model ARE5

In [34]:
dotenv.load_dotenv()
CDS_url, CDS_key = os.getenv('CDS_URL'), os.getenv('CDS_API_KEY')
config_path = Path.home() / '.cdsapirc'
if not config_path.exists():
    print("Creating .cdsapirc ...")
    config_path.write_text(f"url: {CDS_url}\nkey: {CDS_key}\n", encoding='utf-8')
    print("Created at:", config_path)

In [55]:
variables = [
    'total_precipitation', # Precipitation
    '2m_temperature', # Temperature
    '10m_u_component_of_wind', '10m_v_component_of_wind', # Wind
    '2m_dewpoint_temperature',  # Humidity
    'surface_solar_radiation_downwards', # Radiation
]
dataset = 'reanalysis-era5-single-levels'
forcing_dir = os.path.join(test_folder, 'data/forcing')
if not os.path.exists(forcing_dir): os.makedirs(forcing_dir)
start, end = '2025-01-01 00:00:00', '2025-01-10 00:00:00'
start_time = datetime.strptime(start, '%Y-%m-%d %H:%M:%S')
end_time = datetime.strptime(end, '%Y-%m-%d %H:%M:%S')
minx, miny, maxx, maxy = catchment.total_bounds
area = [round(float(maxy), 2), round(float(minx), 2), round(float(miny), 2), round(float(maxx), 2)]

In [61]:
client = cdsapi.Client()
# Download ERA5 data monthly
current = start_time.replace(day=1)
while current <= end_time:
    year, month = current.year, current.month
    last_day = calendar.monthrange(year, month)[1]
    month_start = datetime(year, month, 1)
    month_end = datetime(year, month, last_day, 23)
    # Clip by requested range
    actual_start = max(start_time, month_start)
    actual_end = min(end_time, month_end)
    # Days to download
    days = [f"{d:02d}" for d in range(actual_start.day, actual_end.day + 1)]
    # Output file
    out_file = f"ERA5_{year}_{month:02d}.nc"
    output = os.path.join(forcing_dir, out_file)
    # Skip existing file
    if os.path.exists(output): os.remove(output)
    print(f"Downloading: {out_file}")
    request = {
        'product_type': 'reanalysis', 'variable': ['total_precipitation'],
        'year': '2025', 'month': '01', 'day': '02',
        'time': ['01:00'], 'area': area, 'format': 'netcdf'
    }
    client.retrieve(dataset, request, output)
    # Next month
    current += relativedelta(months=1)

2026-05-14 23:24:49,814 INFO [2026-05-14T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 19 May. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure/14954).


Downloading: ERA5_2025_01.nc


2026-05-14 23:24:52,404 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-05-14 23:24:52,405 INFO Request ID is bba491dd-f6b8-4d6e-8247-6d7e8461426f
2026-05-14 23:24:52,503 INFO status has been updated to accepted
2026-05-14 23:25:14,414 INFO status has been updated to running
2026-05-14 23:25:25,874 INFO status has been updated to successful


d5bafef5f7dbfd77d998f9b5bcfc5587.nc:   0%|          | 0.00/25.0k [00:00<?, ?B/s]

In [63]:
ds = xr.open_dataset(r"test\data\forcing\ERA5_2025_01.nc")

In [64]:
ds

<xarray.Dataset> Size: 52B
Dimensions:     (valid_time: 1, latitude: 1, longitude: 1)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 8B 2025-01-02T01:00:00
  * latitude    (latitude) float64 8B 62.5
  * longitude   (longitude) float64 8B 6.5
    number      int64 8B ...
    expver      <U4 16B ...
Data variables:
    tp          (valid_time, latitude, longitude) float32 4B ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-05-14T21:25 GRIB to CDM+CF via cfgrib-0.9.1...

In [ ]:
# rename variables
ds = ds.rename({
    "tp": "precip",
    "t2m": "temp",
    "u10": "u_wind",
    "v10": "v_wind"
})

ds["precip"] = ds["precip"] * 1000  # m → mm
ds["temp"] = ds["temp"] - 273.15

ds = ds.rio.write_crs(4326)
ds = ds.rio.clip_box(
    minx=6.3,
    miny=62.0,
    maxx=6.8,
    maxy=62.5
)
ds = ds.interp(
    latitude=ds.latitude,
    longitude=ds.longitude,
    method="linear"
)

encoding = {var: {"zlib": True, "complevel": 4} for var in ds.data_vars}

ds.to_netcdf("forcing.nc", encoding=encoding)

def era5_to_hydromt(model, start, end):

    bbox = model.region.to_crs(4326).total_bounds
    area = [bbox[3], bbox[0], bbox[1], bbox[2]]

    out_raw = "era5_raw.nc"
    out_final = "forcing.nc"

    download_era5(area, start, end, out_raw)

    ds = xr.open_dataset(out_raw)

    ds = ds.rename({"tp":"precip","t2m":"temp"})
    ds["precip"] *= 1000
    ds["temp"] -= 273.15

    ds = ds.rio.clip_box(*bbox)

    ds.to_netcdf(out_final)

    return out_final

In [ ]:
# # Write forcing

# def create_forcing(out_path:str, terrain:rioxarray, time:np.array, 
#     values:np.ndarray, variable_name:str, unit:str):
#     crs = terrain.rio.crs
#     if crs is None: raise ValueError("Terrain has no crs")
#     ny, nx = terrain.rio.height, terrain.rio.width
#     data_3d = np.ones((len(time), ny, nx), dtype=np.float32) * values[:, np.newaxis, np.newaxis]
#     da = xr.DataArray(
#         data=data_3d, dims=('time', 'y', 'x'), 
#         coords={'time': time, 'y': terrain.y, 'x': terrain.x},
#         name=variable_name, attrs={'units': unit}
#     )
#     da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
#     da.rio.write_crs(crs, inplace=True)
#     da.rio.write_coordinate_system(inplace=True)
#     da.rio.write_grid_mapping(inplace=True)
#     ds = da.to_dataset()
#     ds[variable_name].attrs["grid_mapping"] = "spatial_ref"
#     comp = {"zlib": True, "complevel": 4, "shuffle": True, "chunksizes": (1, 256, 256)}
#     encoding = {variable_name: comp}
#     ds.to_netcdf(out_path, engine='netcdf4', encoding=encoding)

# weather_path = os.path.join(sample_folder, 'alesund_weather.csv')
# weather = pd.read_csv(weather_path, parse_dates=['datetime'], index_col='datetime')
# weather_new = weather.loc['2025-01-01 00:00:00':'2025-01-10 00:00:00']
# # Write forcing for precipitation
# precip_path = f"{test_folder}/data/forcing/precipitation.nc"
# create_forcing(
#     precip_path, terrain, weather_new.index.to_numpy(), 
#     weather_new['precip_mm'].values, 'precip', '(mm/h)'
# )

In [ ]:
# Clip dtm to catchment
catchment_UTM = catchment.to_crs(terrain.rio.crs)
terrain_clipped = clip_catchment(catchment_UTM, terrain)
terrain_out_path = os.path.normpath(os.path.join(f'{test_folder}/data/dtm', "dtm_clipped.tif"))
terrain_clipped.rio.to_raster(terrain_out_path)

In [14]:
# Fix invalid soil polygon
soil_UTM = soil.to_crs(terrain.rio.crs)
soil_cols = ['soil', 'theta_s', 'theta_r', 'k_sat_ver', 'soil_depth', 'conductivity_decay', 'brooks_corey']
soil_UTM = fix_invalid_polygon(soil_UTM, soil_cols)
soil_layers = ['theta_s', 'theta_r', 'k_sat_ver', 'soil_depth', 'conductivity_decay', 'brooks_corey']
for value in soil_layers:
    soil_path = os.path.normpath(os.path.join('test/data/soil', f'{value}.tif'))
    write_tif(soil_path, terrain, soil_UTM, value)

In [8]:
# Fix invalid land polygon
land_UTM = land.to_crs(terrain.rio.crs)
land_cols = ['land', 'LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
land_UTM = fix_invalid_polygon(land_UTM, land_cols)
land_layers = ['LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
for value in land_layers:
    land_path = os.path.normpath(os.path.join('test/data/landuse', f'{value}.tif'))
    write_tif(land_path, terrain, land_UTM, value)

In [30]:
# Create raster and lookup table (used for calibration)
land_class = land.to_crs(terrain.rio.crs).copy()
land_class['class'] = None
columns, table = np.unique(land_class['land'].values), {}
for id, item in enumerate(columns):
    temp = land_class[land_class['land'] == item]
    table[item] = np.float32(temp.iloc[0][land_layers].values)
    land_class.loc[temp.index, 'class'] = id
land_class = land_class[['class', 'geometry']]
land_class_path = os.path.normpath(os.path.join('test/data/lookup', 'land_classes.tif'))
write_tif(land_class_path, terrain, land_class, 'class')
# Create lookup table
lookup = pd.DataFrame.from_dict(table, orient='index', columns=land_layers)
lookup.index.name = 'landcover'
lookup.reset_index(inplace=True)
lookup.insert(0, 'class_id', lookup.index)
# Save lookup table to csv
lookup_csv_path = os.path.normpath(os.path.join('test/data/lookup', 'lookup_land.csv'))
lookup.to_csv(lookup_csv_path, index=False)

In [75]:
# Process river
river_UTM = river.to_crs(terrain.rio.crs)
cols = {'width': (0.05, 2), 'depth': (1, 5), 'manning_n': (0.03, 0.06)}
river_cols = river_UTM.columns.drop('geometry', errors='ignore')
for col in river_cols:
    river_UTM[col] = pd.to_numeric(river_UTM[col], errors='coerce')
for col, (low, high) in cols.items():
    river_mask = river_UTM[col].isna() | (river_UTM[col] == 'None')
    river_UTM.loc[river_mask, col] = np.round(np.random.uniform(low, high, river_mask.sum()), 3)
river_UTM = river_UTM[river_UTM.is_valid].reset_index(drop=True)
river_UTM["geometry"] = river_UTM.geometry.apply(lambda g: force_2d(g))
river_dict = {
    'river': '', 'river_width': 'width', 'river_depth': 'depth', 'river_n': 'manning_n'
}
for key, value in river_dict.items():
    river_path = os.path.normpath(os.path.join(f'{test_folder}/data/river', f'{key}.tif'))
    write_tif(river_path, terrain, river_UTM, value)

In [228]:
# Run HydroMT
model_path = os.path.normpath(f'{test_folder}/model')
if os.path.exists(model_path): shutil.rmtree(model_path)
!hydromt build wflow_sbm "./test/model" -i "./test/build.yml" -d "./test/config.yml" -v

2026-05-14 20:48:45,808 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-14 20:48:45,864 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from ./test/config.yml
2026-05-14 20:48:45,905 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.1).
2026-05-14 20:48:45,905 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Envs\hyd_ai\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-05-14 20:48:45,927 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-05-14 20:48:45,927 - hydromt.hydromt_wflow.components.config - config - INFO - Reading default config file from C:/Envs/hyd_ai/Lib/site-packages/hydromt_wflow/data/wflow_sbm/wflow_sbm.toml.
2026-05-14 20:48:45,927 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-14 20:48:45,927 - hydromt.model.model - model - INFO - build: setup_config
2026-05-14 20:48:45,927 - hydromt.m

In [ ]:

weather

,datetime,precip_mm,temp_C,shortwave_Wm2,longwave_Wm2,wind_mps,relhum_pct,pressure
0,2025-01-01 00:00:00,2.447,5.172,0.000,320.125,7.620,86.733,1009.946
1,2025-01-01 01:00:00,2.383,5.172,129.410,384.656,7.993,74.095,987.605
2,2025-01-01 02:00:00,3.192,5.172,250.000,384.239,8.049,70.175,997.114
3,2025-01-01 03:00:00,0.759,5.172,353.553,364.490,7.289,87.534,988.409
4,2025-01-01 04:00:00,1.788,5.172,433.013,322.710,8.959,91.852,1013.289
...,...,...,...,...,...,...,...,...
8755,2025-12-31 19:00:00,1.529,5.000,0.000,343.680,5.803,71.896,990.648
8756,2025-12-31 20:00:00,0.724,5.000,0.000,301.795,9.012,93.424,1017.078
8757,2025-12-31 21:00:00,0.415,5.000,0.000,378.202,9.810,85.269,1017.871
8758,2025-12-31 22:00:00,1.135,5.000,0.000,360.407,9.215,84.581,1012.472
